# Week 51, Day 4: How fast should the prior adapt?

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w051/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.signal import lfilter
d = np.load(dataset("ibl_biased.npz"))
mouse, session = d["mouse"].astype(int), d["session"].astype(int)
contrast, choice, reward = d["contrast"].astype(float), d["choice"].astype(float), d["reward"].astype(int)
p_left, rt = d["p_left"].astype(float).round(2), d["rt"].astype(float)
ok = (choice >= 0) & d["included"] & np.isfinite(rt)
sess = 10 * mouse + session
f_c = np.tanh(5 * contrast) / np.tanh(5)
c_pm = np.where(choice >= 0, 2 * choice - 1, 0.0)
side = np.where(contrast != 0, np.sign(contrast), np.where(reward == 1, c_pm, -c_pm))
biased = p_left != 0.5

def kernel(x, alpha, sess_ids):
    out = np.zeros(len(x))
    for s in np.unique(sess_ids):
        k = sess_ids == s
        out[k] = lfilter([alpha], [1, -(1 - alpha)], np.r_[0, x[k][:-1]])
    return out

def bayes_prior(side, sess_ids, h=0.02):
    out = np.zeros(len(side))
    for s in np.unique(sess_ids):
        b = np.array([0.5, 0.5])
        for t in np.flatnonzero(sess_ids == s):
            pr = 0.8 * b[0] + 0.2 * b[1]; out[t] = np.log(pr / (1 - pr))
            b = b * (np.array([0.8, 0.2]) if side[t] > 0 else np.array([0.2, 0.8])); b = b / b.sum()
            b = np.array([(1 - h) * b[0] + h * b[1], (1 - h) * b[1] + h * b[0]])
    return out

prior_bayes = bayes_prior(side, sess)
p_action, p_bayes, alphas = np.zeros(len(choice)), np.zeros(len(choice)), np.zeros(18)
for m in range(18):
    k = mouse == m; use = ok[k]; y = choice[k][use]
    def fit(make):
        f = lambda p: np.sum(np.logaddexp(0, make(p)[use]) - y * make(p)[use])
        return f
    act = lambda p: p[0] * f_c[k] + p[1] + p[2] * kernel(c_pm[k], p[3], sess[k])
    bay = lambda p: p[0] * f_c[k] + p[1] + p[2] * prior_bayes[k]
    pa = minimize(fit(act), [3, 0, 1, 0.2], method="L-BFGS-B", bounds=[(-20, 20)] * 3 + [(0.01, 1)]).x
    pb = minimize(fit(bay), [3, 0, 1], method="L-BFGS-B", bounds=[(-20, 20)] * 3).x
    p_action[k], p_bayes[k], alphas[m] = 1 / (1 + np.exp(-act(pa))), 1 / (1 + np.exp(-bay(pb))), pa[3]
print(f"mice's kernel rates: median {np.median(alphas):.2f}, range {alphas.min():.2f} to {alphas.max():.2f}")

### Exercise: The rate that predicts the next stimulus best

For each alpha in grid, compute the stimulus kernel, fit one weight w by minimising the mean negative log-likelihood of y (1 for a right stimulus) over biased trials, and store the minimum in loss. Then compute loss_bayes, the same quantity with prior_bayes used directly as the log-odds.

In [ ]:
grid = np.array([0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5])
y = (side > 0).astype(float)[biased]
loss = np.zeros(len(grid))
for i, a in enumerate(grid):
    K = kernel(side, a, sess)[biased]
    loss[i] = ...
best_alpha = grid[np.argmin(loss)]
z = prior_bayes[biased]
loss_bayes = ...
print("loss per trial:", {float(a): round(float(v), 4) for a, v in zip(grid, loss)})
print(f"best rate {best_alpha} (memory about {1 / best_alpha:.0f} trials); the mice's median {np.median(alphas):.2f}")
print(f"Bayesian observer: {loss_bayes:.4f}, better than the best leaky average by {loss.min() - loss_bayes:.4f} nats per trial")
plt.plot(grid, loss, "o-", label="leaky average"); plt.axhline(loss_bayes, color="k", ls="--", label="Bayesian observer")
for a in alphas:
    plt.axvline(a, color="tab:pink", alpha=0.25)
plt.xlabel("rate alpha"); plt.ylabel("loss per trial (nats)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_curve():
    "loss has one value per rate in grid, each below log 2"
    assert loss.shape == grid.shape and np.all(loss < np.log(2))
def test_best():
    "best_alpha minimises loss and lies between 0.15 and 0.35"
    assert best_alpha == grid[np.argmin(loss)] and 0.15 <= best_alpha <= 0.35
def test_bayes():
    "the Bayesian observer predicts better than the best leaky average"
    assert loss_bayes < loss.min()

test_curve()
test_best()
test_bayes()
print("All checks passed")

### Exercise: Switch-aligned choices

For every switch between biased blocks in every session, and every lag from -5 to 30, collect hard usable trials. For each, record whether the mouse chose the new block's likely side, and each model's probability of that choice. Average them at each lag into curves (columns: data, action model, Bayesian model).

In [ ]:
hard = ok & (np.abs(contrast) <= 0.0625)
lags = np.arange(-5, 31)
rows, n_sw = [], 0
for s in np.unique(sess):
    idx = np.flatnonzero(sess == s); p = p_left[idx]
    switches = np.flatnonzero((p[1:] != p[:-1]) & (p[:-1] != 0.5)) + 1
    n_sw += len(switches)
    for j in switches:
        new_right = p[j] == 0.2
        for lg in lags:
            t = j + lg
            if 0 <= t < len(idx) and hard[idx[t]]:
                i = idx[t]
                rows.append(...)
rows = np.array(rows, dtype=float)
curves = np.array([rows[rows[:, 0] == lg, 1:].mean(0) for lg in lags])
print(f"{len(rows)} hard trials around {n_sw} block switches")
for lg in [-5, -1, 0, 1, 2, 5, 10, 20, 30]:
    print(f"lag {lg:3d}: data {curves[lags == lg, 0][0]:.2f}, action {curves[lags == lg, 1][0]:.2f}, Bayes {curves[lags == lg, 2][0]:.2f}")
plt.plot(lags, curves[:, 0], "o", label="mice"); plt.plot(lags, curves[:, 1], label="action kernel"); plt.plot(lags, curves[:, 2], label="Bayesian observer")
plt.axvline(0, color="grey"); plt.xlabel("trials since switch"); plt.ylabel("P(choose new likely side)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "curves has one row per lag from -5 to 30 and three columns"
    assert curves.shape == (36, 3)
def test_data():
    "before the switch, mice favour the old side; 20 trials after, the new one"
    assert curves[lags == -1, 0][0] < 0.35 and curves[lags == 20, 0][0] > 0.65
def test_models():
    "both models track the data within 0.1 at every lag from 0 to 20"
    k = (lags >= 0) & (lags <= 20)
    assert np.abs(curves[k, 1] - curves[k, 0]).max() < 0.1 and np.abs(curves[k, 2] - curves[k, 0]).max() < 0.1

test_shape()
test_data()
test_models()
print("All checks passed")

### Exercise: A ladder of reward rates

On usable zero-contrast trials in biased blocks, compute the fraction rewarded for each policy: r_oracle (the block's likely side), r_bayes (the sign of prior_bayes), r_kernel (the sign of each mouse's own action kernel, already computed in ka) and r_mice (the actual reward).

In [ ]:
ka = np.zeros(len(choice))
for m in range(18):
    k = mouse == m
    ka[k] = kernel(c_pm[k], alphas[m], sess[k])
z = ok & (contrast == 0) & biased
r_oracle = ...
r_bayes = ...
r_kernel = ...
r_mice = ...
for name, v in [("oracle", r_oracle), ("Bayesian observer", r_bayes), ("own action kernel", r_kernel), ("mice", r_mice)]:
    print(f"{name:18s} {v:.1%}")

In [ ]:
import numpy as np
def test_values():
    "the four reward rates are computed on usable zero-contrast trials in biased blocks"
    z = ok & (contrast == 0) & biased
    assert abs(r_mice - reward[z].mean()) < 1e-12 and abs(r_oracle - np.mean(np.where(p_left[z] == 0.2, 1, -1) == side[z])) < 1e-12
def test_order():
    "oracle, Bayes, own kernel and mice are in decreasing order"
    assert r_oracle > r_bayes > r_kernel > r_mice

test_values()
test_order()
print("All checks passed")